# 03. ONNX Backtest (2024)

This notebook runs out-of-time backtest on 2024 data using the exported ONNX model.
It mirrors feature logic from `02_feature_engineering_and_train.ipynb`.


In [1]:
from datetime import datetime
import numpy as np
import polars as pl

try:
    import onnxruntime as rt
except ImportError as e:
    raise ImportError("onnxruntime is required. Install with: pip install onnxruntime") from e


In [2]:
# Config
DATA_PATH = "../data/processed/aggregated_demand_3min.parquet"
ONNX_PATH = "../models/taxi_demand_model.onnx"

# Must match training-time setup in notebook 02
FEATURE_MODE = "lite"  # "lite" or "full"
HORIZON_STEPS = 5       # 3min step * 5 = 15min ahead

# Backtest window (inclusive start, exclusive end)
BACKTEST_START = datetime(2024, 10, 1)
BACKTEST_END = datetime(2025, 1, 1)

# Inference memory control
CHUNK_SIZE = 500_000

# PASS/FAIL thresholds
MIN_MAE_IMPROVEMENT_PCT = 5.0
MIN_RMSE_IMPROVEMENT_PCT = 3.0
MAX_ABS_BIAS = 0.10


In [3]:
# Load ONNX session
sess = rt.InferenceSession(ONNX_PATH)
onnx_inputs = sess.get_inputs()
onnx_output_name = sess.get_outputs()[0].name

print("ONNX model loaded:", ONNX_PATH)
print("Input count:", len(onnx_inputs))
for inp in onnx_inputs:
    print(f"  - name={inp.name}, type={inp.type}, shape={inp.shape}")
print("Output:", onnx_output_name)


ONNX model loaded: ../models/taxi_demand_model.onnx
Input count: 1
  - name=float_input, type=tensor(float), shape=[None, 5]
Output: variable


In [4]:
# Load data
demand_df = pl.read_parquet(DATA_PATH)
demand_df = demand_df.with_columns(pl.col("pickup_time").cast(pl.Datetime)).sort(["PULocationID", "pickup_time"])

print("Data shape:", demand_df.shape)
print("pickup_time range:", demand_df.select(pl.col("pickup_time").min()).item(), "~", demand_df.select(pl.col("pickup_time").max()).item())


Data shape: (131978648, 3)
pickup_time range: 2023-01-01 00:00:00 ~ 2025-11-30 00:00:00


In [5]:
# Feature engineering (same chain as notebook 02)
feature_df = demand_df.with_columns([
    pl.col("pickup_time").dt.hour().alias("hour"),
    pl.col("pickup_time").dt.weekday().alias("day_of_week"),
    (pl.col("pickup_time").dt.weekday() >= 5).cast(pl.Int8).alias("is_weekend"),
    pl.col("demand").shift(1).over("PULocationID").alias("demand_lag_1"),
    pl.col("demand").shift(5).over("PULocationID").alias("demand_lag_5"),
    pl.col("demand").shift(20).over("PULocationID").alias("demand_lag_20"),
    pl.col("demand").rolling_mean(window_size=20).over("PULocationID").alias("rolling_mean_20"),
    pl.col("demand").shift(-HORIZON_STEPS).over("PULocationID").alias("target")
])

base_features = ["PULocationID", "hour", "day_of_week", "is_weekend"]
lite_features = base_features + ["demand_lag_20"]
full_features = base_features + ["demand_lag_1", "demand_lag_5", "demand_lag_20", "rolling_mean_20"]

if FEATURE_MODE == "full":
    features = full_features
elif FEATURE_MODE == "lite":
    features = lite_features
else:
    raise ValueError("FEATURE_MODE must be 'lite' or 'full'")

required_cols = features + ["target"]
feature_df = feature_df.drop_nulls(subset=required_cols)

print("Feature mode:", FEATURE_MODE)
print("Features:", features)
print("Feature df shape:", feature_df.shape)


Feature mode: lite
Features: ['PULocationID', 'hour', 'day_of_week', 'is_weekend', 'demand_lag_20']
Feature df shape: (131972147, 11)


In [6]:
# Backtest split
test_df = feature_df.filter(
    (pl.col("pickup_time") >= BACKTEST_START) &
    (pl.col("pickup_time") < BACKTEST_END)
)

print("Backtest window:", BACKTEST_START, "~", BACKTEST_END)
print("Test samples:", test_df.height)
if test_df.height == 0:
    raise ValueError("No samples found in backtest window. Check date range and data coverage.")

print("Actual test range:", test_df.select(pl.col("pickup_time").min()).item(), "~", test_df.select(pl.col("pickup_time").max()).item())


Backtest window: 2024-10-01 00:00:00 ~ 2025-01-01 00:00:00
Test samples: 11481600
Actual test range: 2024-10-01 00:00:00 ~ 2024-12-31 23:57:00


In [7]:
def _is_int_input(onnx_input_type: str) -> bool:
    return "int64" in onnx_input_type.lower() or "int32" in onnx_input_type.lower()


def _run_onnx_predict_chunked(df: pl.DataFrame, feature_names: list[str], session, output_name: str, chunk_size: int = 500_000) -> np.ndarray:
    n = df.height
    pred = np.empty(n, dtype=np.float32)
    inputs = session.get_inputs()

    # Case A: single tensor input (N, F)
    if len(inputs) == 1:
        input_name = inputs[0].name
        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            batch = df.slice(start, end - start).select(feature_names)
            x_np = batch.to_numpy().astype(np.float32)
            out = session.run([output_name], {input_name: x_np})[0]
            pred[start:end] = np.array(out).reshape(-1).astype(np.float32)
        return pred

    # Case B: per-feature named inputs
    input_type_map = {x.name: x.type for x in inputs}
    missing = [f for f in feature_names if f not in input_type_map]
    if missing:
        raise ValueError(f"ONNX input mismatch. Missing feature inputs: {missing}")

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        batch = df.slice(start, end - start).select(feature_names)
        feed = {}
        for name in feature_names:
            arr = batch[name].to_numpy().reshape(-1, 1)
            if _is_int_input(input_type_map[name]):
                feed[name] = arr.astype(np.int64)
            else:
                feed[name] = arr.astype(np.float32)
        out = session.run([output_name], feed)[0]
        pred[start:end] = np.array(out).reshape(-1).astype(np.float32)

    return pred


In [8]:
# ONNX inference
print("Running ONNX inference...")
y_pred = _run_onnx_predict_chunked(test_df, features, sess, onnx_output_name, chunk_size=CHUNK_SIZE)
y_true = test_df["target"].to_numpy().astype(np.float32)
y_naive = test_df["demand"].to_numpy().astype(np.float32)

mae = float(np.mean(np.abs(y_true - y_pred)))
rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
bias = float(np.mean(y_pred - y_true))

naive_mae = float(np.mean(np.abs(y_true - y_naive)))
naive_rmse = float(np.sqrt(np.mean((y_true - y_naive) ** 2)))
naive_bias = float(np.mean(y_naive - y_true))

mae_improve_pct = (naive_mae - mae) / naive_mae * 100.0 if naive_mae > 0 else np.nan
rmse_improve_pct = (naive_rmse - rmse) / naive_rmse * 100.0 if naive_rmse > 0 else np.nan

print("\n--- Overall Metrics ---")
print(f"Model MAE  : {mae:.4f}")
print(f"Model RMSE : {rmse:.4f}")
print(f"Model Bias : {bias:.4f}")
print(f"Naive MAE  : {naive_mae:.4f}")
print(f"Naive RMSE : {naive_rmse:.4f}")
print(f"Naive Bias : {naive_bias:.4f}")
print(f"MAE Improvement vs Naive  : {mae_improve_pct:.2f}%")
print(f"RMSE Improvement vs Naive : {rmse_improve_pct:.2f}%")


Running ONNX inference...

--- Overall Metrics ---
Model MAE  : 0.3754
Model RMSE : 1.3226
Model Bias : -0.1688
Naive MAE  : 0.4346
Naive RMSE : 1.4166
Naive Bias : -0.0001
MAE Improvement vs Naive  : 13.61%
RMSE Improvement vs Naive : 6.63%


In [10]:
# Build evaluation dataframe
# 1) Add `pred` first
eval_df = test_df.select([
    "PULocationID", "pickup_time", "hour", "day_of_week", "is_weekend", "demand", "target"
]).with_columns(
    pl.Series("pred", y_pred)
)

# 2) Add derived columns that reference `pred`
eval_df = eval_df.with_columns([
    (pl.col("pred") - pl.col("target")).alias("err"),
    (pl.col("pred") - pl.col("target")).abs().alias("abs_err"),
    ((pl.col("pred") - pl.col("target")) ** 2).alias("sq_err"),
    (pl.col("demand") - pl.col("target")).alias("naive_err"),
    (pl.col("demand") - pl.col("target")).abs().alias("naive_abs_err"),
    ((pl.col("demand") - pl.col("target")) ** 2).alias("naive_sq_err"),
])

zone_stats = eval_df.group_by("PULocationID").agg([
    pl.col("target").mean().alias("mean_target"),
    pl.col("abs_err").mean().alias("mae"),
    pl.col("sq_err").mean().sqrt().alias("rmse"),
    pl.col("err").mean().alias("bias"),
    pl.col("naive_abs_err").mean().alias("naive_mae"),
    pl.col("naive_sq_err").mean().sqrt().alias("naive_rmse")
])

q20 = zone_stats.select(pl.col("mean_target").quantile(0.2)).item()
q80 = zone_stats.select(pl.col("mean_target").quantile(0.8)).item()
hot_zones = zone_stats.filter(pl.col("mean_target") >= q80)
low_zones = zone_stats.filter(pl.col("mean_target") <= q20)

def _zone_summary(df: pl.DataFrame, label: str):
    return {
        "label": label,
        "zones": int(df.height),
        "mae": float(df.select(pl.col("mae").mean()).item()),
        "rmse": float(df.select(pl.col("rmse").mean()).item()),
        "bias": float(df.select(pl.col("bias").mean()).item()),
        "naive_mae": float(df.select(pl.col("naive_mae").mean()).item()),
        "naive_rmse": float(df.select(pl.col("naive_rmse").mean()).item()),
    }

hot = _zone_summary(hot_zones, "hot(top20%)")
low = _zone_summary(low_zones, "low(bottom20%)")

print("\n--- Zone Split Metrics ---")
print(hot)
print(low)

print("\nWorst 10 zones by MAE")
print(zone_stats.sort("mae", descending=True).head(10))



--- Zone Split Metrics ---
{'label': 'hot(top20%)', 'zones': 53, 'mae': 1.7408898386134384, 'rmse': 2.5982391946689054, 'bias': -0.730296426515302, 'naive_mae': 1510243052.7489767, 'naive_rmse': 2.8312390514705514}
{'label': 'low(bottom20%)', 'zones': 53, 'mae': 0.0011821831246959158, 'rmse': 0.028838380979764044, 'bias': -0.001163297517488423, 'naive_mae': 4796898.290839487, 'naive_rmse': 0.04067403723462543}

Worst 10 zones by MAE
shape: (10, 7)
┌──────────────┬─────────────┬──────────┬──────────┬───────────┬───────────┬────────────┐
│ PULocationID ┆ mean_target ┆ mae      ┆ rmse     ┆ bias      ┆ naive_mae ┆ naive_rmse │
│ ---          ┆ ---         ┆ ---      ┆ ---      ┆ ---       ┆ ---       ┆ ---        │
│ i64          ┆ f64         ┆ f64      ┆ f64      ┆ f64       ┆ f64       ┆ f64        │
╞══════════════╪═════════════╪══════════╪══════════╪═══════════╪═══════════╪════════════╡
│ 132          ┆ 10.920584   ┆ 3.847905 ┆ 5.165885 ┆ -0.442208 ┆ 1.8728e9  ┆ 5.527505   │
│ 237  

In [11]:
# Time bucket metrics (commute/weekend/night)
time_df = eval_df.with_columns(
    pl.when((pl.col("hour") >= 7) & (pl.col("hour") <= 9)).then(pl.lit("commute_morning"))
    .when((pl.col("hour") >= 17) & (pl.col("hour") <= 20)).then(pl.lit("commute_evening"))
    .when(pl.col("is_weekend") == 1).then(pl.lit("weekend"))
    .when((pl.col("hour") >= 0) & (pl.col("hour") <= 5)).then(pl.lit("late_night"))
    .otherwise(pl.lit("other"))
    .alias("time_bucket")
)

bucket_stats = time_df.group_by("time_bucket").agg([
    pl.col("abs_err").mean().alias("mae"),
    pl.col("sq_err").mean().sqrt().alias("rmse"),
    pl.col("err").mean().alias("bias"),
    pl.col("naive_abs_err").mean().alias("naive_mae"),
    pl.col("naive_sq_err").mean().sqrt().alias("naive_rmse"),
    pl.len().alias("n")
]).with_columns([
    ((pl.col("naive_mae") - pl.col("mae")) / pl.col("naive_mae") * 100).alias("mae_improve_pct"),
    ((pl.col("naive_rmse") - pl.col("rmse")) / pl.col("naive_rmse") * 100).alias("rmse_improve_pct")
]).sort("time_bucket")

print("\n--- Time Bucket Metrics ---")
print(bucket_stats)



--- Time Bucket Metrics ---
shape: (5, 9)
┌────────────┬──────────┬──────────┬───────────┬───┬────────────┬─────────┬────────────┬───────────┐
│ time_bucke ┆ mae      ┆ rmse     ┆ bias      ┆ … ┆ naive_rmse ┆ n       ┆ mae_improv ┆ rmse_impr │
│ t          ┆ ---      ┆ ---      ┆ ---       ┆   ┆ ---        ┆ ---     ┆ e_pct      ┆ ove_pct   │
│ ---        ┆ f64      ┆ f64      ┆ f64       ┆   ┆ f64        ┆ u32     ┆ ---        ┆ ---       │
│ str        ┆          ┆          ┆           ┆   ┆            ┆         ┆ f64        ┆ f64       │
╞════════════╪══════════╪══════════╪═══════════╪═══╪════════════╪═════════╪════════════╪═══════════╡
│ commute_ev ┆ 0.513371 ┆ 1.638435 ┆ -0.221583 ┆ … ┆ 1.756778   ┆ 1913600 ┆ 100.0      ┆ 6.736358  │
│ ening      ┆          ┆          ┆           ┆   ┆            ┆         ┆            ┆           │
│ commute_mo ┆ 0.371564 ┆ 1.174583 ┆ -0.164225 ┆ … ┆ 1.250438   ┆ 1435200 ┆ 100.0      ┆ 6.066279  │
│ rning      ┆          ┆          ┆           ┆

In [15]:
# Top-K zone ranking metrics (model vs naive)
TOP_K_LIST = [3, 5, 10]

rank_df = eval_df.select(["pickup_time", "PULocationID", "target", "pred", "demand"]).with_columns([
    pl.col("target").rank("ordinal", descending=True).over("pickup_time").alias("actual_rank"),
    pl.col("pred").rank("ordinal", descending=True).over("pickup_time").alias("model_rank"),
    pl.col("demand").rank("ordinal", descending=True).over("pickup_time").alias("naive_rank"),
])

all_ts = rank_df.select("pickup_time").unique()
topk_results = []

for k in TOP_K_LIST:
    actual_topk = rank_df.filter(pl.col("actual_rank") <= k).select(["pickup_time", "PULocationID"])
    model_topk = rank_df.filter(pl.col("model_rank") <= k).select(["pickup_time", "PULocationID"])
    naive_topk = rank_df.filter(pl.col("naive_rank") <= k).select(["pickup_time", "PULocationID"])

    actual_cnt = actual_topk.group_by("pickup_time").len().rename({"len": "actual_cnt"})

    model_hit_cnt = actual_topk.join(model_topk, on=["pickup_time", "PULocationID"], how="inner").group_by("pickup_time").len().rename({"len": "hit_cnt"})
    model_ts = all_ts.join(actual_cnt, on="pickup_time", how="left").join(model_hit_cnt, on="pickup_time", how="left").with_columns([
        pl.col("actual_cnt").fill_null(0),
        pl.col("hit_cnt").fill_null(0),
    ])

    naive_hit_cnt = actual_topk.join(naive_topk, on=["pickup_time", "PULocationID"], how="inner").group_by("pickup_time").len().rename({"len": "hit_cnt"})
    naive_ts = all_ts.join(actual_cnt, on="pickup_time", how="left").join(naive_hit_cnt, on="pickup_time", how="left").with_columns([
        pl.col("actual_cnt").fill_null(0),
        pl.col("hit_cnt").fill_null(0),
    ])

    model_hit_rate = float(model_ts.select((pl.col("hit_cnt") > 0).cast(pl.Float64).mean()).item())
    model_precision = float(model_ts.select((pl.col("hit_cnt") / k).mean()).item())
    model_recall = float(model_ts.select((pl.col("hit_cnt") / pl.when(pl.col("actual_cnt") > 0).then(pl.col("actual_cnt")).otherwise(1)).mean()).item())

    naive_hit_rate = float(naive_ts.select((pl.col("hit_cnt") > 0).cast(pl.Float64).mean()).item())
    naive_precision = float(naive_ts.select((pl.col("hit_cnt") / k).mean()).item())
    naive_recall = float(naive_ts.select((pl.col("hit_cnt") / pl.when(pl.col("actual_cnt") > 0).then(pl.col("actual_cnt")).otherwise(1)).mean()).item())

    topk_results.append({
        "k": k,
        "model_hit_rate": model_hit_rate,
        "model_precision_at_k": model_precision,
        "model_recall_at_k": model_recall,
        "naive_hit_rate": naive_hit_rate,
        "naive_precision_at_k": naive_precision,
        "naive_recall_at_k": naive_recall,
    })

topk_df = pl.DataFrame(topk_results)
print("\n--- Top-K Ranking Metrics (Model vs Naive) ---")
print(topk_df)



--- Top-K Ranking Metrics (Model vs Naive) ---
shape: (3, 7)
┌─────┬───────────────┬───────────────┬───────────────┬──────────────┬──────────────┬──────────────┐
│ k   ┆ model_hit_rat ┆ model_precisi ┆ model_recall_ ┆ naive_hit_ra ┆ naive_precis ┆ naive_recall │
│ --- ┆ e             ┆ on_at_k       ┆ at_k          ┆ te           ┆ ion_at_k     ┆ _at_k        │
│ i64 ┆ ---           ┆ ---           ┆ ---           ┆ ---          ┆ ---          ┆ ---          │
│     ┆ f64           ┆ f64           ┆ f64           ┆ f64          ┆ f64          ┆ f64          │
╞═════╪═══════════════╪═══════════════╪═══════════════╪══════════════╪══════════════╪══════════════╡
│ 3   ┆ 0.886005      ┆ 0.489485      ┆ 0.489485      ┆ 0.86284      ┆ 0.464591     ┆ 0.464591     │
│ 5   ┆ 0.977264      ┆ 0.539198      ┆ 0.539198      ┆ 0.963474     ┆ 0.503954     ┆ 0.503954     │
│ 10  ┆ 0.998822      ┆ 0.5999        ┆ 0.5999        ┆ 0.997826     ┆ 0.563524     ┆ 0.563524     │
└─────┴───────────────┴──────

In [13]:
# Spike detection metrics (model vs naive)
# Spike definition: per-zone target >= zone 90th percentile
SPIKE_Q = 0.90

zone_thr = eval_df.group_by("PULocationID").agg(
    pl.col("target").quantile(SPIKE_Q).alias("zone_spike_thr")
)

spike_df = eval_df.join(zone_thr, on="PULocationID", how="left").with_columns([
    (pl.col("target") >= pl.col("zone_spike_thr")).cast(pl.Int8).alias("actual_spike"),
    (pl.col("pred") >= pl.col("zone_spike_thr")).cast(pl.Int8).alias("model_spike"),
    (pl.col("demand") >= pl.col("zone_spike_thr")).cast(pl.Int8).alias("naive_spike"),
])

def _binary_metrics(df: pl.DataFrame, pred_col: str, actual_col: str = "actual_spike"):
    tp = float(df.filter((pl.col(pred_col) == 1) & (pl.col(actual_col) == 1)).height)
    fp = float(df.filter((pl.col(pred_col) == 1) & (pl.col(actual_col) == 0)).height)
    fn = float(df.filter((pl.col(pred_col) == 0) & (pl.col(actual_col) == 1)).height)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
    }

model_spike = _binary_metrics(spike_df, "model_spike")
naive_spike = _binary_metrics(spike_df, "naive_spike")

print("\n--- Spike Detection Metrics (Model vs Naive) ---")
print({"spike_quantile": SPIKE_Q, "model": model_spike, "naive": naive_spike})



--- Spike Detection Metrics (Model vs Naive) ---
{'spike_quantile': 0.9, 'model': {'precision': 0.9997253532726541, 'recall': 0.9214106225132875, 'f1': 0.9589717455834167, 'tp': 8430335, 'fp': 2316, 'fn': 719044}, 'naive': {'precision': 0.9751784444483792, 'recall': 0.975174927172653, 'f1': 0.9751766858073446, 'tp': 8922245, 'fp': 227101, 'fn': 227134}}


In [14]:
# PASS / FAIL gate
checks = {
    "MAE improvement >= threshold": bool(mae_improve_pct >= MIN_MAE_IMPROVEMENT_PCT),
    "RMSE improvement >= threshold": bool(rmse_improve_pct >= MIN_RMSE_IMPROVEMENT_PCT),
    "Absolute bias <= threshold": bool(abs(bias) <= MAX_ABS_BIAS),
    "Hot-zone MAE beats naive": bool(hot["mae"] <= hot["naive_mae"]),
}
overall_pass = all(checks.values())

print("\n--- PASS/FAIL ---")
for k, v in checks.items():
    print(f"{k}: {'PASS' if v else 'FAIL'}")
print("Overall:", "PASS" if overall_pass else "FAIL")



--- PASS/FAIL ---
MAE improvement >= threshold: PASS
RMSE improvement >= threshold: PASS
Absolute bias <= threshold: FAIL
Hot-zone MAE beats naive: PASS
Overall: FAIL
